In [ ]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from matplotlib import rcParams

In [ ]:
# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from PIL import Image

In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Bengali Handwritten Character Recognition using Ekush Dataset

In [ ]:
# Display PyTorch version and GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Step 1: Define paths and parameters
# You'll need to update these paths according to your dataset location
BASE_PATH = r''  # Update this path
METADATA_PATH = os.path.join(BASE_PATH, r"C:\Users\KIIT\Desktop\minor\metaData_img.csv")  # Update if filename is different
IMG_SIZE = 28  # We'll resize all images to 28x28
BATCH_SIZE = 28
EPOCHS = 15
NUM_CLASSES = 122  # 122 Bengali character classes (0-121)

In [ ]:
# Step 2: Load metadata
# The metadata file maps folder names to actual Bengali characters
metadata = pd.read_csv(METADATA_PATH)
print("Metadata information:")
print(metadata.head())

# Create a mapping from class index to Bengali character name
class_to_char = dict(zip(metadata['Folder Name'].astype(int), metadata['Char Name']))
print(f"Sample mapping - Class 0: {class_to_char[0]}")

In [ ]:
# Step 3: Function to load and preprocess images
def load_images(base_path, img_size=IMG_SIZE):
    """
    Load images from the directory structure and preprocess them.
    Assumes images are already grayscale with white characters on black background.

    Args:
        base_path: Path to the dataset
        img_size: Size to resize images to

    Returns:
        X: Image data as numpy array
        y: Labels as numpy array
    """
    X = []
    y = []

    # Loop through each class folder
    for class_idx in range(NUM_CLASSES):
        class_dir = os.path.join(base_path, str(class_idx))
        print(f"Loading class {class_idx}: {class_to_char.get(class_idx, 'Unknown')}")

        # Check if directory exists
        if not os.path.exists(class_dir):
            print(f"Warning: Directory for class {class_idx} not found at {class_dir}")
            continue

        # Get all image files in the class directory
        try:
            img_files = [f for f in os.listdir(class_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
            print(f"Found {len(img_files)} images in class {class_idx}")
        except Exception as e:
            print(f"Error reading directory {class_dir}: {e}")
            continue

        # Optional: Limit the number of images per class for faster development
        # Uncomment for quicker experimentation
        # img_files = img_files[:500]  # Using 500 images per class for faster development

        for img_file in img_files:
            img_path = os.path.join(class_dir, img_file)

            try:
                # Read and preprocess image
                img = Image.open(img_path).convert('L')  # Load as grayscale
                img = img.resize((img_size, img_size))
                
                # Convert to array and normalize
                img_array = np.array(img) / 255.0  # Normalize to [0, 1]

                # Verify image isn't just noise by checking variance
                if np.var(img_array) > 0.01:  # Adjust threshold as needed
                    X.append(img_array)
                    y.append(class_idx)
                else:
                    print(f"Skipping possible noisy image: {img_path}")

            except Exception as e:
                print(f"Error processing image {img_path}: {e}")
                continue

    # Convert lists to numpy arrays
    X = np.array(X)
    y = np.array(y)

    # Reshape X to [N, 1, H, W] for PyTorch (add channel dimension)
    X = X.reshape(-1, 1, img_size, img_size)

    print(f"Successfully loaded {len(X)} images with shape {X.shape}")
    print(f"Labels shape: {y.shape}")

    # Add a check to visualize some loaded images to verify correct loading
    if len(X) > 0:
        plt.figure(figsize=(10, 5))
        for i in range(min(5, len(X))):
            plt.subplot(1, 5, i+1)
            plt.imshow(X[i].reshape(img_size, img_size), cmap='gray')
            plt.title(f"Class: {y[i]}")
            plt.axis('off')
        plt.tight_layout()
        plt.show()

    return X, y

In [ ]:
# Step 4: Load and prepare data
X, y = load_images(BASE_PATH)

In [ ]:
# Step 5: Exploratory Data Analysis
# Visualize some sample images
plt.figure(figsize=(12, 8))
for i in range(9):
    plt.subplot(3, 3, i+1)
    # Display the image
    plt.imshow(X[i].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
    # Add title with class name
    plt.title(f"Class: {y[i]} ({class_to_char.get(y[i], 'Unknown')})")
    plt.axis('off')
plt.tight_layout()
plt.savefig('sample_bengali_characters.png')
plt.show()

# Check class distribution
plt.figure(figsize=(12, 6))
sns.countplot(x=y)
plt.title('Distribution of Classes')
plt.xlabel('Class Index')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('class_distribution.png')
plt.show()

In [ ]:
# Step 6: Split the dataset
# Split into train, validation, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

In [ ]:
# Step 7: Create PyTorch Dataset and DataLoader classes
class BengaliCharDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        # Convert to PyTorch tensors
        image = torch.FloatTensor(image)
        
        # Apply transforms if any
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Define transforms for data augmentation
train_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.RandomAffine(degrees=0, scale=(0.9, 1.1))
])

# Create datasets
train_dataset = BengaliCharDataset(X_train, y_train, transform=train_transform)
val_dataset = BengaliCharDataset(X_val, y_val)
test_dataset = BengaliCharDataset(X_test, y_test)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
# Step 8: Visualize data augmentation examples
def visualize_augmentation(dataset, n_samples=9):
    plt.figure(figsize=(12, 8))
    
    for i in range(n_samples):
        # Get a random sample
        idx = np.random.randint(0, len(dataset))
        img, label = dataset[idx]
        
        # Convert tensor to numpy array for visualization
        img_np = img.numpy().squeeze()
        
        plt.subplot(3, 3, i+1)
        plt.imshow(img_np, cmap='gray')
        plt.title(f"Augmented: Class {label}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig('augmented_samples.png')
    plt.show()

# Visualize augmented samples
visualize_augmentation(train_dataset)

In [ ]:
# Step 9: Define the CNN Model
class BengaliCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(BengaliCNN, self).__init__()
        
        # First convolutional block
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.2)
        )
        
        # Second convolutional block
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.3)
        )
        
        # Third convolutional block
        self.conv3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.4)
        )
        
        # Calculate size after three pooling layers
        # Starting with 28x28, after 3 pooling layers: 28/(2^3) = 3.5, so 3x3
        # For exact calculations, we'd need to trace through the layers
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.fc(x)
        return x

# Create the model
model = BengaliCNN(NUM_CLASSES).to(device)

# Display model summary
print(model)

In [ ]:
# Step 10: Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.2, patience=3, min_lr=0.00001, verbose=True)

In [ ]:
# Step 11: Training loop
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=EPOCHS):
    # Initialize best validation accuracy
    best_val_acc = 0.0
    
    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    # Early stopping parameters
    patience = 5
    early_stop_counter = 0
    
    # Training loop
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Track training statistics
            train_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
        
        # Calculate epoch loss and accuracy
        epoch_train_loss = train_loss / train_total
        epoch_train_acc = train_correct / train_total
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                
                # Forward pass
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                # Track validation statistics
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        # Calculate validation loss and accuracy
        epoch_val_loss = val_loss / val_total
        epoch_val_acc = val_correct / val_total
        
        # Update learning rate scheduler
        scheduler.step(epoch_val_loss)
        
        # Save model if validation accuracy improved
        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            torch.save(model.state_dict(), 'best_bengali_char_model.pth')
            early_stop_counter = 0
        else:
            early_stop_counter += 1
        
        # Print epoch statistics
        print(f'Epoch {epoch+1}/{epochs}: '
              f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f}, '
              f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}')
        
        # Update history
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        # Early stopping
        if early_stop_counter >= patience:
            print(f'Early stopping after {epoch+1} epochs')
            break
    
    # Load best model weights
    model.load_state_dict(torch.load('best_bengali_char_model.pth'))
    
    return model, history

# Train the model
model, history = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, EPOCHS)

In [ ]:
# Step 12: Visualize training history
plt.figure(figsize=(12, 5))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history['train_acc'], label='Training Accuracy')
plt.plot(history['val_acc'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history['train_loss'], label='Training Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()

In [ ]:
# Step 13: Evaluate on test set
def evaluate_model(model, dataloader, criterion):
    model.eval()
    
    test_loss = 0.0
    test_correct = 0
    test_total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Track test statistics
            test_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            test_total += labels.size(0)
            test_correct += (predicted == labels).sum().item()
            
            # Save predictions and labels for classification report
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate test loss and accuracy
    test_loss = test_loss / test_total
    test_acc = test_correct / test_total
    
    print(f"Test accuracy: {test_acc:.4f}")
    print(f"Test loss: {test_loss:.4f}")
    
    return test_acc, test_loss, all_preds, all_labels

# Evaluate model on test set
test_acc, test_loss, y_pred, y_true = evaluate_model(model, test_loader, criterion)

In [ ]:
# Step 14: Classification report
print("Classification Report:")
class_report = classification_report(y_true, y_pred)
print(class_report)

# Save the classification report to file
with open('classification_report.txt', 'w') as f:
    f.write(class_report)

In [ ]:
# Step 15: Confusion Matrix (for a subset of classes)
# We'll visualize a subset due to the large number of classes
plt.figure(figsize=(12, 10))
subset_size = 20  # Visualize first 20 classes
subset_indices = np.where((np.array(y_true) < subset_size) & (np.array(y_pred) < subset_size))[0]

cm_subset = confusion_matrix(
    np.array(y_true)[subset_indices],
    np.array(y_pred)[subset_indices],
    labels=range(subset_size)
)

# Normalize confusion matrix
cm_normalized = cm_subset.astype('float') / cm_subset.sum(axis=1)[:, np.newaxis]

# Create heatmap
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=[class_to_char.get(i, str(i)) for i in range(subset_size)],
    yticklabels=[class_to_char.get(i, str(i)) for i in range(subset_size)]
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (First 20 Classes)')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

In [ ]:
# Step 16: Visualize some model predictions
def visualize_predictions(X_data, y_true, y_pred, num_samples=10):
    """Visualize model predictions alongside true labels"""
    
    # Get random indices
    indices = np.random.choice(len(X_data), num_samples, replace=False)
    
    plt.figure(figsize=(15, 6))
    for i, idx in enumerate(indices):
        plt.subplot(2, 5, i+1)
        plt.imshow(X_data[idx].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
        
        true_label = class_to_char.get(y_true[idx], str(y_true[idx]))
        pred_label = class_to_char.get(y_pred[idx], str(y_pred[idx]))
        
        title = f"True: {true_label}\nPred: {pred_label}"
        color = "green" if y_true[idx] == y_pred[idx] else "red"
        
        plt.title(title, color=color)
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig('prediction_samples.png')
    plt.show()

# Convert predictions to numpy arrays for visualization
y_true_np = np.array(y_true)
y_pred_np = np.array(y_pred)

# Visualize predictions
visualize_predictions(X_test, y_true_np, y_pred_np)

In [ ]:
# Step 17: Save the model for future use
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'NUM_CLASSES': NUM_CLASSES,
    'IMG_SIZE': IMG_SIZE
}, r"C:\Users\KIIT\Desktop\minor\ekkush_recognition_model_pytorch.pth")

print("Model saved successfully!")

In [ ]:
# Step 18: Create a simple prediction function for new images
def predict_bengali_character(img_path, model):
    """
    Predict the Bengali character from a given image file.
    
    Args:
        img_path: Path to the image file
        model: Trained model
        
    Returns:
        predicted_class: The predicted class index
        confidence: Confidence score of the prediction
        character: Bengali character (if mapping available)
    """
    # Ensure model is in evaluation mode
    model.eval()
    
    # Load and preprocess the image
    img = Image.open(img_path).convert('L')  # Load as grayscale
    img = img.resize((IMG_SIZE, IMG_SIZE))
    
    # Convert to tensor
    img_tensor = torch.FloatTensor(np.array(img) / 255.0)
    img_tensor = img_tensor.view(1, 1, IMG_SIZE, IMG_SIZE).to(device)
    
    # Make prediction
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.nn.functional.softmax(output, dim=1)[0]
        
    # Get predicted class and confidence
    predicted_class = torch.argmax(probabilities).item()
    confidence = probabilities[predicted_class].item()
    
    # Get the Bengali character if available
    character = class_to_char.get(predicted_class, "Unknown")
    
    # Display results
    plt.figure(figsize=(4, 4))
    plt.imshow(img, cmap='gray')
    plt.title(f"Predicted: {character} (Class {predicted_class})\nConfidence: {confidence:.4f}")
    plt.axis('off')
    plt.show()
    
    return predicted_class, confidence, character

In [ ]:
# Example usage (uncomment and update path when needed)
# test_img_path = 'path/to/test/image.jpg'
# predicted_class, confidence, character = predict_bengali_character(test_img_path, model)
# print(f"Predicted character: {character} (Class {predicted_class}) with {confidence:.2%} confidence")

plt.rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Noto Sans Bengali', 'Nirmala UI', 'Bangla Sangam MN', 'Lohit Bengali']

# Function to display images with Bengali text
def plot_bengali_predictions(images, true_labels, pred_labels, bengali_chars_dict, rows=2, cols=5, figsize=(15, 8)):
    """
    Plot images with Bengali labels
    
    Parameters:
    images: array of images to display
    true_labels: true class labels (numeric)
    pred_labels: predicted class labels (numeric)
    bengali_chars_dict: dictionary mapping class numbers to Bengali characters
    """
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    
    for i, ax in enumerate(axes):
        if i < len(images):
            ax.imshow(images[i].reshape(IMG_SIZE, IMG_SIZE), cmap='gray')
            ax.axis('off')
            
            true_char = bengali_chars_dict.get(true_labels[i], str(true_labels[i]))
            pred_char = bengali_chars_dict.get(pred_labels[i], str(pred_labels[i]))
            
            # Set title with both correct and predicted labels
            ax.set_title(f'True: {true_char}\nPred: {pred_char}', 
                        color='green' if true_labels[i] == pred_labels[i] else 'red',
                        fontsize=12)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Load model for inference
def load_model_for_inference(model_path):
    # Load the model checkpoint
    checkpoint = torch.load(model_path)
    
    # Create a new model with the same architecture
    model = BengaliCNN(num_classes=checkpoint['NUM_CLASSES']).to(device)
    
    # Load the state dict
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Set model to evaluation mode
    model.eval()
    
    return model

In [ ]:
# Load and test the model on a new image
model_path = r"C:\Users\KIIT\Desktop\minor\ekkush_recognition_model_pytorch.pth"
loaded_model = load_model_for_inference(model_path)

In [ ]:
# Test on a new image
test_img_path = r"C:\Users\KIIT\Desktop\minor\Screenshot 2025-04-09 054801.png"  # Replace with actual path
predicted_class, confidence, character = predict_bengali_character(test_img_path, loaded_model)
print(f"Predicted character: {character} (Class {predicted_class}) with {confidence:.2%} confidence")